In [14]:
# !pip install -qU crewai[tools]==1.15.21

In [15]:
# !pip install -qU tavily-python scrapegraph-py

In [16]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
import agentops
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List, Optional
from tavily import TavilyClient
from scrapegraph_py import ScrapeGraphAI as ScrapeGraphClient

import os
import json

load_dotenv(override=True)
os.environ.pop("GROQ_API_KEY", None)

In [17]:
required_env_vars = [
    "GOOGLE_API_KEY",
    "GOOGLE_MODEL",
    "AGENTOPS_API_KEY",
    "TVLY_API_KEY",
    "SCRAPEGRAPH_API_KEY",
]

missing_env_vars = [name for name in required_env_vars if not os.getenv(name)]
if missing_env_vars:
    raise RuntimeError(f"Missing environment variables: {', '.join(missing_env_vars)}")

agentops.init(
    api_key=os.environ["AGENTOPS_API_KEY"],
    skip_auto_end_session=True,
    default_tags=["crewai"],
 )

In [18]:
print(agentops.get_client().config.exporter_endpoint)

https://otlp.agentops.ai/v1/traces


In [19]:
output_dir = "./ai-agent-output"
os.makedirs(output_dir, exist_ok=True)

basic_llm = LLM(
    model=f"google/{os.environ['GOOGLE_MODEL']}",
    api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0,
 )
search_client = TavilyClient(api_key=os.environ["TVLY_API_KEY"])
scrape_client = ScrapeGraphClient(api_key=os.environ["SCRAPEGRAPH_API_KEY"])

In [20]:
no_keywords = 10

about_company = "Rankyx is a company that provides AI solutions to help websites refine their search and recommendation systems."

## Setup Agents

### Agent: A

In [21]:
class SuggestedSearchQueries(BaseModel):
    queries: List[str] = Field(..., title="Suggested search queries to be passed to the search engine",
                                min_length=1, max_length=no_keywords)

search_queries_recommendation_agent = Agent(
    role="Search Queries Recommendation Agent",
    goal="\n".join([
                "To provide a list of suggested search queries to be passed to the search engine.",
                "The queries must be varied and looking for specific items."
            ]),
    backstory="The agent is designed to help in looking for products by providing a list of suggested search queries to be passed to the search engine based on the context provided.",
    llm=basic_llm,
    verbose=True,
)

search_queries_recommendation_task = Task(
    description="\n".join([
        "Rankyx is looking to buy {product_name} at the best prices (value for a price strategy)",
        "The campany target any of these websites to buy from: {websites_list}",
        "The company wants to reach all available proucts on the internet to be compared later in another stage.",
        "The stores must sell the product in {country_name}",
        "Generate at maximum {no_keywords} queries.",
        "The search keywords must be in {language} language.",
        "Search keywords must contains specific brands, types or technologies. Avoid general keywords.",
        "The search query must reach an ecommerce webpage for product, and not a blog or listing page."
    ]),
    expected_output="A JSON object containing a list of suggested search queries.",
    output_json=SuggestedSearchQueries,
    output_file=os.path.join(output_dir, "step_1_suggested_search_queries.json"),
    agent=search_queries_recommendation_agent
)

### Agent: B

In [22]:
class SignleSearchResult(BaseModel):
    title: str
    url: str = Field(..., title="the page url")
    content: str
    score: float
    search_query: str

class AllSearchResults(BaseModel):
    results: List[SignleSearchResult]

@tool
def search_engine_tool(query: str):
    """Useful for search-based queries. Use this to find current information about any query related pages using a search engine"""
    return search_client.search(query)

search_engine_agent = Agent(
    role="Search Engine Agent",
    goal="To search for products based on the suggested search query",
    backstory="The agent is designed to help in looking for products by searching for products based on the suggested search queries.",
    llm=basic_llm,
    verbose=True,
    tools=[search_engine_tool]
)

search_engine_task = Task(
    description="\n".join([
        "The task is to search for products based on the suggested search queries.",
        "You have to collect results from multiple search queries.",
        "Ignore any susbicious links or not an ecommerce single product website link.",
        "Ignore any search results with confidence score less than ({score_th}) .",
        "The search results will be used to compare prices of products from different websites.",
    ]),
    expected_output="A JSON object containing the search results.",
    output_json=AllSearchResults,
    output_file=os.path.join(output_dir, "step_2_search_results.json"),
    agent=search_engine_agent
)

### Agent: C

In [23]:
class ProductSpec(BaseModel):
    specification_name: str
    specification_value: str

class SingleExtractedProduct(BaseModel):
    page_url: str = Field(..., title="The original url of the product page")
    product_title: str = Field(..., title="The title of the product")
    product_image_url: str = Field(..., title="The url of the product image")
    product_url: str = Field(..., title="The url of the product")
    product_current_price: float = Field(..., title="The current price of the product")
    product_original_price: Optional[float] = Field(title="The original price of the product before discount. Set to None if no discount", default=None)
    product_discount_percentage: Optional[float] = Field(title="The discount percentage of the product. Set to None if no discount", default=None)

    product_specs: List[ProductSpec] = Field(..., title="The specifications of the product. Focus on the most important specs to compare.", min_length=1, max_length=5)

    agent_recommendation_rank: int = Field(..., title="The rank of the product to be considered in the final procurement report. (out of 5, Higher is Better) in the recommendation list ordering from the best to the worst")
    agent_recommendation_notes: List[str]  = Field(..., title="A set of notes why would you recommend or not recommend this product to the company, compared to other products.")


class AllExtractedProducts(BaseModel):
    products: List[SingleExtractedProduct]


@tool
def web_scraping_tool(page_url: str):
    """
    An AI Tool to help an agent to scrape a web page

    Example:
    web_scraping_tool(
        page_url="https://www.noon.com/egypt-en/15-bar-fully-automatic-espresso-machine-1-8-l-1500"
    )
    """
    result = scrape_client.extract(
        prompt="Extract the product details from this page.",
        url=page_url,
        schema=SingleExtractedProduct.model_json_schema(),
    )

    return {
        "page_url": page_url,
        "details": result.data.json_data if result.data else result.error,
    }

scraping_agent = Agent(
    role="Web scraping agent",
    goal="To extract details from any website",
    backstory="The agent is designed to help in looking for required values from any website url. These details will be used to decide which best product to buy.",
    llm=basic_llm,
    tools=[web_scraping_tool],
    verbose=True,
 )

scraping_task = Task(
    description="\n".join([
        "The task is to extract product details from any ecommerce store page url.",
        "The task has to collect results from multiple pages urls.",
        "Collect the best {top_recommendations_no} products from the search results.",
    ]),
    expected_output="A JSON object containing products details",
    output_json=AllExtractedProducts,
    output_file=os.path.join(output_dir, "step_3_search_results.json"),
    agent=scraping_agent
 )

### Agent: D

In [24]:
procurement_report_author_agent = Agent(
    role="Procurement Report Author Agent",
    goal="To generate a professional HTML page for the procurement report",
    backstory="The agent is designed to assist in generating a professional HTML page for the procurement report after looking into a list of products.",
    llm=basic_llm,
    verbose=True,
)

procurement_report_author_task = Task(
    description="\n".join([
        "The task is to generate a professional HTML page for the procurement report.",
        "You have to use Bootstrap CSS framework for a better UI.",
        "Use the provided context about the company to make a specialized report.",
        "The report will include the search results and prices of products from different websites.",
        "The report should be structured with the following sections:",
        "1. Executive Summary: A brief overview of the procurement process and key findings.",
        "2. Introduction: An introduction to the purpose and scope of the report.",
        "3. Methodology: A description of the methods used to gather and compare prices.",
        "4. Findings: Detailed comparison of prices from different websites, including tables and charts.",
        "5. Analysis: An analysis of the findings, highlighting any significant trends or observations.",
        "6. Recommendations: Suggestions for procurement based on the analysis.",
        "7. Conclusion: A summary of the report and final thoughts.",
        "8. Appendices: Any additional information, such as raw data or supplementary materials.",
    ]),

    expected_output="A professional HTML page for the procurement report.",
    output_file=os.path.join(output_dir, "step_4_procurement_report.html"),
    agent=procurement_report_author_agent,
)

## Run the AI Crew

In [25]:
rankyx_crew = Crew(
    agents=[
        search_queries_recommendation_agent,
        search_engine_agent,
        scraping_agent,
        procurement_report_author_agent,
    ],
    tasks=[
        search_queries_recommendation_task,
        search_engine_task,
        scraping_task,
        procurement_report_author_task,
    ],
    process=Process.sequential,
 )

In [26]:
crew_results = await rankyx_crew.kickoff_async(
    inputs={
        "product_name": "coffee machine for the office",
        "websites_list": ["www.amazon.eg", "www.jumia.com.eg", "www.noon.com/egypt-en"],
        "country_name": "Egypt",
        "no_keywords": 10,
        "language": "English",
        "score_th": 0.10,
        "top_recommendations_no": 10
    }
)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Search Queries Recommendation Agent                                                                     │
│                                                                                                                 │
│  Task: Rankyx is looking to buy coffee machine for the office at the best prices (value for a price strategy)   │
│  The campany target any of these websites to buy from: ['www.amazon.eg', 'www.jumia.com.eg',                    │
│  'www.noon.com/egypt-en']                                                                                       │
│  The company wants to reach all available proucts on the internet to be compared later in another stage.        │
│  The stores must sell the product in Egypt                                                                      │
│  Generate at maximum 10 queries.                                                                                │
│  The search keywords must be in English language.                                                               │
│  Search keywords must contains specific brands, types or technologies. Avoid general keywords.                  │
│  The search query must reach an ecommerce webpage for product, and not a blog or listing page.                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Search Queries Recommendation Agent                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  queries=['DeLonghi Magnifica S automatic coffee machine site:amazon.eg', 'Nespresso Vertuo Next coffee         │
│  machine site:noon.com/egypt-en', 'Tornado programmable coffee maker 12 cups site:jumia.com.eg', 'Philips       │
│  Series 2200 espresso machine site:amazon.eg', 'Black and Decker 12 cup coffee maker site:noon.com/egypt-en',   │
│  'Miele CM 5310 coffee machine site:amazon.eg', 'Kenwood drip coffee maker 1.5 liter site:jumia.com.eg',        │
│  'Breville Barista Max espresso machine site:noon.com/egypt-en', 'Fresh coffee maker 1000 watt                  │
│  site:jumia.com.eg', 'Saeco Lirika automatic coffee machine site:amazon.eg']                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Search Engine Agent                                                                                     │
│                                                                                                                 │
│  Task: The task is to search for products based on the suggested search queries.                                │
│  You have to collect results from multiple search queries.                                                      │
│  Ignore any susbicious links or not an ecommerce single product website link.                                   │
│  Ignore any search results with confidence score less than (0.1) .                                              │
│  The search results will be used to compare prices of products from different websites.                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Web scraping agent                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "products": [                                                                                                │
│      {                                                                                                          │
│        "page_url": "https://www.amazon.eg/DeLonghi-Magnifica-Automatic-Coffee-Machine/dp/B00400OMU0",           │
│        "product_title": "De'Longhi Magnifica S Automatic Coffee Machine, ECAM 22.110.B, Black",                 │
│        "product_image_url": "https://m.media-amazon.com/images/I/61+DeLonghi-Magnifica-S.jpg",                  │
│        "product_url": "https://www.amazon.eg/DeLonghi-Magnifica-Automatic-Coffee-Machine/dp/B00400OMU0",        │
│        "product_current_price": 24999.0,                                                                        │
│        "product_original_price": null,                                                                          │
│        "product_discount_percentage": null,                                                                     │
│        "product_specs": [                                                                                       │
│          {                                                                                                      │
│            "specification_name": "Type",                                                                        │
│            "specification_value": "Automatic"                                                                   │
│          },                                                                                                     │
│          {                                                                                                      │
│            "specification_name": "Grinder",                                                                     │
│            "specification_value": "Integrated"                                                                  │
│          },                                                                                                     │
│          {                                                                                                      │
│            "specification_name": "Milk System",                                                                 │
│            "specification_value": "Steam wand"                                                                  │
│          }                                                                                                      │
│        ],                                                                                                       │
│        "agent_recommendation_rank": 5,                                                                          │
│        "agent_recommendation_notes": [                                                                          │
│          "Highly recommended for home use due to its balance of features and price.",                           │
│          "Integrated grinder provides fresh coffee experience."                                                 │
│        ]                                                                                                        │
│      },                                                

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Procurement Report Author Agent                                                                         │
│                                                                                                                 │
│  Task: The task is to generate a professional HTML page for the procurement report.                             │
│  You have to use Bootstrap CSS framework for a better UI.                                                       │
│  Use the provided context about the company to make a specialized report.                                       │
│  The report will include the search results and prices of products from different websites.                     │
│  The report should be structured with the following sections:                                                   │
│  1. Executive Summary: A brief overview of the procurement process and key findings.                            │
│  2. Introduction: An introduction to the purpose and scope of the report.                                       │
│  3. Methodology: A description of the methods used to gather and compare prices.                                │
│  4. Findings: Detailed comparison of prices from different websites, including tables and charts.               │
│  5. Analysis: An analysis of the findings, highlighting any significant trends or observations.                 │
│  6. Recommendations: Suggestions for procurement based on the analysis.                                         │
│  7. Conclusion: A summary of the report and final thoughts.                                                     │
│  8. Appendices: Any additional information, such as raw data or supplementary materials.                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Procurement Report Author Agent                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  <!DOCTYPE html>                                                                                                │
│  <html lang="en">                                                                                               │
│  <head>                                                                                                         │
│      <meta charset="UTF-8">                                                                                     │
│      <meta name="viewport" content="width=device-width, initial-scale=1.0">                                     │
│      <title>Procurement Report: Coffee Machine Market Analysis</title>                                          │
│      <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">     │
│      <style>                                                                                                    │
│          body { background-color: #f8f9fa; }                                                                    │
│          .report-container { background: #ffffff; padding: 40px; border-radius: 8px; box-shadow: 0 0 10px       │
│  rgba(0,0,0,0.1); margin-top: 20px; }                                                                           │
│          h1, h2, h3 { color: #2c3e50; }                                                                         │
│          .table-responsive { margin-top: 20px; }                                                                │
│      </style>                                                                                                   │
│  </head>                                                                                                        │
│  <body>                                                                                                         │
│                                                                                                                 │
│  <div class="container report-container">                                                                       │
│      <header class="text-center mb-5">                                                                          │
│          <h1>Procurement Report: Coffee Machine Market Analysis</h1>                                            │
│          <p class="text-muted">Date: October 2023 | Prepared by: Procurement Agent</p>                          │
│      </header>                                                                                                  │
│                                                                                                                 │
│      <section id="executive-summary">                                                                           │
│          <h2>1. Executive Summary</h2>                                                                          │
│          <p>This report evaluates the current market landscape for coffee machines across major Egyptian        │
│  e-commerce platforms (Amazon.eg, Noon.com, and Jumia.com.eg). Our findings indicate a wide price variance      │
│  ranging from budget-friendly drip coffee makers (EGP 1,450) to premium automatic espresso systems (EGP         │
│  65,000). We recommend a tiered procurement strategy based on specific departmental needs.</p>                  │
│      </section>                                        

┌───────────────────────── Trace Batch Finalization ──────────────────────────┐
│ ✅ Trace batch finalized with session ID:                                   │
│ 855242da-2758-4f77-af35-831f1c5e95c8                                        │
│                                                                             │
│ 🔗 View here:                                                               │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/855242da-2758-4f │
│ 77-af35-831f1c5e95c8?access_code=TRACE-5892bca762                           │
│ 🔑 Access Code: TRACE-5892bca762                                            │
└─────────────────────────────────────────────────────────────────────────────┘
